In [1]:
#Loading the Excel File and libraries
import pandas as pd
import numpy as np

file_path = '/content/Global Oil and Gas Extraction Tracker - 2024-02-22_1039.xlsx'
xl = pd.ExcelFile(file_path)

In [2]:
#setting dataframes with main, P&R
main_data_df = xl.parse('Main data')
production_reserves_df = xl.parse('Production & reserves')


In [3]:
# Filtering for 2022 and 2023 data
filtered_main_data_df = main_data_df[main_data_df["Discovery year"].isin([2022, 2023])]

In [4]:
filtered_main_data_df

,Unit ID,Unit Name,Unit name local script,Fuel type,Unit type,Country,"Subnational unit (province, state)",Latitude,Longitude,Location accuracy,...,FID Year,Production start year,Operator,Owner,Parent,Basin,Concession / block,Project or complex,Government unit ID,Wiki URL
3423,OG0012600,Adrar,NaN,oil and gas,field,Algeria,NaN,25.824000,1.799000,exact,...,NaN,NaN,CNPC,NaN,NaN,NaN,NaN,NaN,NaN,https://www.gem.wiki/Adrar_Oil_and_Gas_Field_%...
7356,OG0017430,HDLE/HDLS,NaN,oil and gas,field,Algeria,NaN,NaN,NaN,NaN,...,NaN,2022,"Eni, Sonatrach",Sonatrach (51%);Eni S.P.A. (49%),Sonatrach SPA (51%);Eni S.P.A. (49%),Berkine,Zemlet el Arbi,NaN,NaN,https://www.gem.wiki/HDLE/HDLS_Oil_and_Gas_Fie...
7397,OG0017480,Narges-1X,NaN,gas,field,Egypt,NaN,31.721267,33.340298,approximate,...,NaN,NaN,Chevron,Chevron (45%);International Egyptian Oil Compa...,Chevron (45%);Eni S.P.A. (45%);Tharwa (10%),NaN,Nargis,NaN,NaN,https://www.gem.wiki/Narges-1X_Gas_Field_%28Eg...
7406,OG0017489,Venus,NaN,oil,field,Namibia,NaN,-29.503000,13.650000,exact,...,2025 (expected),2029 (expected),TotalEnergies,TotalEnergies (45.2%);QatarEnergy (30%);Impact...,TotalEnergies (45.2%);QatarEnergy (30%);Africa...,Orange,Block 2913B,Venus,NaN,https://www.gem.wiki/Venus_Oil_Field_%28Namibi...
7408,OG0017491,Graff,NaN,oil and gas,field,Namibia,NaN,-29.689065,13.939756,exact,...,2027 (expected),2032 (expected),Shell plc,QatarEnergy (45%);Shell plc (45%);National Pet...,QatarEnergy (45%);Shell plc (45%);National Pet...,Orange,Block 2914A,Graff,NaN,https://www.gem.wiki/Graff_Oil_and_Gas_Field_%...
7468,OG0017567,Lau Lau,NaN,oil and gas,field,Guyana,NaN,7.702300,-56.460900,exact,...,NaN,NaN,ExxonMobil,ExxonMobil (45%);Hess Guyana Exploration Ltd (...,ExxonMobil (45%);Hess Corporation (30%);CNOOC ...,NaN,Stabroek Block,NaN,NaN,https://www.gem.wiki/Lau_Lau_Oil_and_Gas_Field...
7469,OG0017568,Fangtooth,NaN,oil and gas,field,Guyana,NaN,7.942200,-57.071000,exact,...,NaN,NaN,ExxonMobil,ExxonMobil (45%);Hess Guyana Exploration Ltd (...,ExxonMobil (45%);Hess Corporation (30%);CNOOC ...,NaN,Stabroek Block,NaN,NaN,https://www.gem.wiki/Fangtooth_Oil_and_Gas_Fie...
7470,OG0017569,Barreleye,NaN,oil and gas,field,Guyana,NaN,7.727000,-56.866000,exact,...,NaN,NaN,ExxonMobil,ExxonMobil (45%);Hess Guyana Exploration Ltd (...,ExxonMobil (45%);Hess Corporation (30%);CNOOC ...,NaN,Stabroek Block,NaN,NaN,https://www.gem.wiki/Barreleye_Oil_and_Gas_Fie...
7471,OG0017572,Seabob,NaN,oil and gas,field,Guyana,NaN,7.732800,-56.753800,exact,...,NaN,NaN,ExxonMobil,ExxonMobil (45%);Hess Guyana Exploration Ltd (...,ExxonMobil (45%);Hess Corporation (30%);CNOOC ...,NaN,Stabroek Block,NaN,NaN,https://www.gem.wiki/Seabob_Oil_and_Gas_Field_...
7472,OG0017574,Sailfin,NaN,oil and gas,field,Guyana,NaN,7.869870,-56.606250,approximate,...,NaN,NaN,ExxonMobil,ExxonMobil (45%);Hess Guyana Exploration Ltd (...,ExxonMobil (45%);Hess Corporation (30%);CNOOC ...,NaN,Stabroek Block,NaN,NaN,https://www.gem.wiki/Sailfin_Oil_and_Gas_Field...


In [5]:
# Convert 'Data year' to integers in the 'production_reserves_df'
production_reserves_df['Data year'] = pd.to_numeric(production_reserves_df['Data year'], errors='coerce').fillna(-1).astype(int)


In [6]:
#Define functions for getting the most recent value, calculate the total reserves, calculate the total production and define the conversion factor
# Function to get the most recent value based on the criteria
def get_most_recent_value(unit_id, prod_res, units, df):
    # Filter based on Unit ID, Production/reserves, and Units (converted)
    filtered = df[
        (df["Unit ID"] == unit_id) &
        (df["Production/reserves"] == prod_res) &
        (df["Units (converted)"] == units)
    ]

    # Sort by Data year and get the most recent entry
    filtered = filtered.sort_values(by="Data year", ascending=False)
    return filtered.iloc[0]["Quantity (converted)"] if not filtered.empty else np.nan

# Function to calculate the total reserves
def calculate_total_reserves(row):
    if pd.notna(row['Reserves- Hydrocarbons (unspecified)']):
        return row['Reserves- Hydrocarbons (unspecified)']
    else:
        # Convert gas reserves to boe
        gas_in_boe = row['Reserves- Gas'] * conversion_factor if pd.notna(row['Reserves- Gas']) else 0
        oil_reserves = row['Reserves - Oil'] if pd.notna(row['Reserves - Oil']) else 0
        return gas_in_boe + oil_reserves

# Function to calculate the total production
def calculate_total_production(row):
    if pd.notna(row['Production - Hydrocarbons (unspecified)']):
        return row['Production - Hydrocarbons (unspecified)']
    else:
        # Convert gas production to boe
        gas_in_boe = row['Production - Gas'] * conversion_factor if pd.notna(row['Production - Gas']) else 0
        oil_production = row['Production - Oil'] if pd.notna(row['Production - Oil']) else 0
        return gas_in_boe + oil_production

# Conversion factor from million m³ to million boe for gas
conversion_factor = 5.883 / 1000


In [7]:
# Populate reserves and production columns with the most recent data
filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
filtered_main_data_df["Reserves- Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million boe", production_reserves_df))

filtered_main_data_df["Production - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million bbl/y", production_reserves_df))
filtered_main_data_df["Production - Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million m³/y", production_reserves_df))
filtered_main_data_df["Production - Hydrocarbons (unspecified)"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "production", "million boe/y", production_reserves_df))

# Calculate total reserves and production
filtered_main_data_df['Reserves- Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_reserves, axis=1)
filtered_main_data_df['Production - Total (Oil, Gas and Hydrocarbons)'] = filtered_main_data_df.apply(calculate_total_production, axis=1)


<ipython-input-7-e0a58fe69be6>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves - Oil"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million bbl", production_reserves_df))
<ipython-input-7-e0a58fe69be6>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_main_data_df["Reserves- Gas"] = filtered_main_data_df["Unit ID"].apply(lambda x: get_most_recent_value(x, "reserves", "million m³", production_reserves_df))
<ipython-input-7-e

In [8]:
filtered_main_data_df

,Unit ID,Unit Name,Unit name local script,Fuel type,Unit type,Country,"Subnational unit (province, state)",Latitude,Longitude,Location accuracy,...,Government unit ID,Wiki URL,Reserves - Oil,Reserves- Gas,Reserves- Hydrocarbons (unspecified),Production - Oil,Production - Gas,Production - Hydrocarbons (unspecified),"Reserves- Total (Oil, Gas and Hydrocarbons)","Production - Total (Oil, Gas and Hydrocarbons)"
3423,OG0012600,Adrar,NaN,oil and gas,field,Algeria,NaN,25.824000,1.799000,exact,...,NaN,https://www.gem.wiki/Adrar_Oil_and_Gas_Field_%...,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
7356,OG0017430,HDLE/HDLS,NaN,oil and gas,field,Algeria,NaN,NaN,NaN,NaN,...,NaN,https://www.gem.wiki/HDLE/HDLS_Oil_and_Gas_Fie...,140.0,NaN,NaN,NaN,NaN,NaN,140.000000,0.000000
7397,OG0017480,Narges-1X,NaN,gas,field,Egypt,NaN,31.721267,33.340298,approximate,...,NaN,https://www.gem.wiki/Narges-1X_Gas_Field_%28Eg...,NaN,99108.02775,NaN,NaN,NaN,NaN,583.052527,0.000000
7406,OG0017489,Venus,NaN,oil,field,Namibia,NaN,-29.503000,13.650000,exact,...,NaN,https://www.gem.wiki/Venus_Oil_Field_%28Namibi...,2000.0,14463.70899,NaN,NaN,NaN,NaN,2085.090000,0.000000
7408,OG0017491,Graff,NaN,oil and gas,field,Namibia,NaN,-29.689065,13.939756,exact,...,NaN,https://www.gem.wiki/Graff_Oil_and_Gas_Field_%...,200.0,155741.18650,149.0,NaN,NaN,NaN,149.000000,0.000000
7468,OG0017567,Lau Lau,NaN,oil and gas,field,Guyana,NaN,7.702300,-56.460900,exact,...,NaN,https://www.gem.wiki/Lau_Lau_Oil_and_Gas_Field...,NaN,NaN,658.0,NaN,NaN,NaN,658.000000,0.000000
7469,OG0017568,Fangtooth,NaN,oil and gas,field,Guyana,NaN,7.942200,-57.071000,exact,...,NaN,https://www.gem.wiki/Fangtooth_Oil_and_Gas_Fie...,NaN,NaN,620.0,NaN,NaN,NaN,620.000000,0.000000
7470,OG0017569,Barreleye,NaN,oil and gas,field,Guyana,NaN,7.727000,-56.866000,exact,...,NaN,https://www.gem.wiki/Barreleye_Oil_and_Gas_Fie...,NaN,NaN,572.0,NaN,NaN,NaN,572.000000,0.000000
7471,OG0017572,Seabob,NaN,oil and gas,field,Guyana,NaN,7.732800,-56.753800,exact,...,NaN,https://www.gem.wiki/Seabob_Oil_and_Gas_Field_...,NaN,NaN,591.0,NaN,NaN,NaN,591.000000,0.000000
7472,OG0017574,Sailfin,NaN,oil and gas,field,Guyana,NaN,7.869870,-56.606250,approximate,...,NaN,https://www.gem.wiki/Sailfin_Oil_and_Gas_Field...,NaN,NaN,563.0,NaN,NaN,NaN,563.000000,0.000000


In [9]:
output_excel_path = '/content/Discovery20222023 20240222.xlsx'
filtered_main_data_df.to_excel(output_excel_path, index=False)